### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7efc42a9a030>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7efc42802930>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year of the movie realeased")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

model_with_structure= model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7efc42a9a030>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7efc42802930>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the

In [9]:
model_with_structure= model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7efc42a9a030>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7efc42802930>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': 'A movie with details..', 'parameters': {'properties': {'title': {'descripti

In [5]:
model.invoke("Provide details about the movie Knives out")

AIMessage(content="**Knives Out (2019)**  \n*A modern whodunnit with a star‑packed ensemble, clever twists, and a nod to classic mystery tropes.*\n\n| Category | Details |\n|----------|---------|\n| **Director** | Rian Johnson |\n| **Screenplay** | Rian Johnson |\n| **Producers** | Rian Johnson, Charles Roven, John Rosengrant |\n| **Music** | Nathan Johnson |\n| **Cinematography** | John Toll |\n| **Editing** | Chris Gill |\n| **Production Companies** | Paramount Pictures, Rian Johnson Productions |\n| **Distributor** | Paramount Pictures |\n| **Runtime** | 130 minutes |\n| **Release Dates** | •\u202fU.S. (theatrical): 23\u202fNov\u202f2019 (limited) <br>•\u202fU.S. (wide): 27\u202fNov\u202f2019 <br>•\u202fInternational releases: 24\u202fDec\u202f2019 (UK) – 10\u202fJan\u202f2020 (Australia) |\n| **Budget** | ~\u202f$35\u202fmillion |\n| **Box Office** | $311\u202fmillion worldwide |\n| **Genre** | Mystery / Thriller / Comedy |\n| **Language** | English |\n| **Country** | United States

In [10]:
response=model_with_structure.invoke("Provide details about the movie Knives Out")
response

Movie(title='Knives Out', year=2019, director='Rian Johnson', rating=8.0)

### Message output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details.."""
    title: str = Field(..., description ='The title of the movie')
    year: int = Field(..., description='The year of the movie')
    director: str = Field(..., description='The director of the movie')
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details abpout the movie Knives out")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants details about the movie "Knives Out". We can use the function to retrieve movie details. We\'ll call the function with director, rating, title, year. The function signature: Movie({director, rating, year, title}) => any. So we need to fill in details for Knives Out. We need to know director: Rian Johnson. Rating: maybe 7.9? The rating out of 10? We can give IMDb rating maybe 7.9. Year: 2019. Title: Knives Out. Provide details. Then output the function call.', 'tool_calls': [{'id': 'fc_c2bec57b-90ca-4b65-9309-2962799b7cd2', 'function': {'arguments': '{"director":"Rian Johnson","rating":7.9,"title":"Knives Out","year":2019}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 163, 'prompt_tokens': 166, 'total_tokens': 329, 'completion_time': 0.180790302, 'completion_tokens_details': {'reasoning_tokens': 122}, 'prompt_time': 0.008088705, 'prompt_tokens_detail

### Nested Structure

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role: str

class MovieDetails(BaseModel):
    title : str
    year  : int
    cast  : list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description='Budget in millions USD')


model_with_structure = model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide details about the movie Casino Royale")
response

MovieDetails(title='Casino Royale', year=2006, cast=[Actor(name='Daniel Craig', role='James Bond'), Actor(name='Eva Green', role='Vesper Lynd'), Actor(name='Mads Mikkelsen', role='Le Chiffre'), Actor(name='Mickey Rourke', role='Raoul Silva')], genres=['Action', 'Adventure', 'Thriller'], budget=150000000.0)


### TypedDict

TypedDict provides a simpler alternative using python's built in typing, ideal when you dont need runtime validation

In [13]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "the year of the movie"]
    director: Annotated[str, ..., "the director of the movie"]
    rating: Annotated[float, ..., "The movies's rating out of 10"]

model_with_structure=model.with_structured_output(MovieDict)
response=model_with_structure.invoke("Please Provide the details of the movie Avengers")
response

{'director': 'Anthony Russo & Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [14]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in rupees")


model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Please Provide the details of the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [15]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses

A data class is a class typiaclly containing mainly data, although there arent really any restrictions. you create it using the @dataclass decorator

In [16]:
import os 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7efc42634500>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7efc426373e0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="Email of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo
)


result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info from :JOHN DOE, johndoe@email.com, 98765432"}]
})

result

{'messages': [HumanMessage(content='Extract info from :JOHN DOE, johndoe@email.com, 98765432', additional_kwargs={}, response_metadata={}, id='83a48ffc-a421-48b3-bc4d-9fb1ec8f8502'),
  AIMessage(content='{"name":"JOHN DOE","email":"johndoe@email.com","phone":"98765432"}', additional_kwargs={'reasoning_content': 'We need to produce JSON according to ContactInfo schema. Must include name, email, phone. The input string: "JOHN DOE, johndoe@email.com, 98765432". Name: JOHN DOE, email: johndoe@email.com, phone: 98765432. Format: compact JSON. Ensure it parses. Output only final JSON object.'}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 234, 'total_tokens': 344, 'completion_time': 0.115480253, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.011327092, 'prompt_tokens_details': None, 'queue_time': 0.313678325, 'total_time': 0.126807345}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_dema

In [20]:
result['structured_response']

ContactInfo(name='JOHN DOE', email='johndoe@email.com', phone='98765432')